# Accent bias in Whisper — free GPU run

Runs the transcription step on a free Colab/Kaggle GPU using the **local**
backend, so no Hugging Face inference credits are needed.

The analysis logic is unchanged: this notebook only produces
`results/transcripts_local.jsonl`, which you download and analyse anywhere.

**Before you start:** Runtime → Change runtime type → Hardware accelerator → **GPU**.

> **Do not mix backends in one analysis.** Transcripts from the hosted API and
> from a local model are not comparable — different builds, precision and
> decoding. Keep them in separate files and analyse them separately. That is
> why this notebook writes `transcripts_local.jsonl`, never `transcripts.jsonl`.


## 1. Check the GPU

If this prints `No GPU`, fix the runtime type before going further — CPU works but is far too slow for a full sample.

In [ ]:
!nvidia-smi || echo "No GPU: Runtime > Change runtime type > GPU"

import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. whisper-large-v3 on CPU will take hours.")

## 2. Get the code

In [ ]:
!git clone --depth 1 https://github.com/zahid111777/Accent-bias-in-speech-recognition.git repo
%cd repo
!ls

## 3. Install dependencies

`torch` is already present on Colab; `transformers` and `accelerate` are what the local backend needs on top of the project's own requirements.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q torch transformers accelerate
# Whisper reads mp3 through torchaudio/ffmpeg; Colab ships ffmpeg already.
!python -c "import transformers, accelerate; print('transformers', transformers.__version__)"


## 4. Upload the audio

Build `audio.zip` on your own machine from the folder `prepare_data` produced:

```bat
powershell Compress-Archive -Path audio\* -DestinationPath audio.zip
```

Then run the cell below and choose that file. The archive is **not** in the
repository: the Speech Accent Archive is distributed under its own terms.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose audio.zip

In [ ]:
!unzip -q -o audio.zip -d audio
!echo "clips per accent:"
!for d in audio/*/; do echo -n "  $(basename $d): "; ls "$d" | wc -l; done

## 5. Transcribe on the GPU

The model loads once and is reused for every clip. No language is forced, so
Whisper auto-detects exactly as the hosted API does — a clip transcribed into
the speaker's first language is a result this study measures, not a bug to
suppress.

Transcripts are appended to the cache as they complete, so an interrupted
session resumes instead of restarting. Try `--limit 5` first.

In [ ]:
!python -m src.run     --audio_dir audio     --out results     --backend local     --transcripts results/transcripts_local.jsonl     --limit 5

In [ ]:
!python -m src.run     --audio_dir audio     --out results     --backend local     --transcripts results/transcripts_local.jsonl

## 6. Check what you got

Read the per-accent counts before reading anything into the numbers.

In [ ]:
import pandas as pd
print(pd.read_csv("results/summary_by_accent.csv")[
    ["accent", "n", "mean_wer", "ci95_low", "ci95_high", "n_wrong_language"]
].to_string(index=False))
print()
print(pd.read_csv("results/run_provenance.csv").to_string(index=False))

## 7. Download the transcripts

The cache is the thing worth keeping: rerun the analysis from it anywhere, with no GPU and no credits.

In [ ]:
!zip -q -r transcripts_local.zip results/transcripts_local.jsonl results/*.csv results/figures
from google.colab import files
files.download("transcripts_local.zip")

## 8. Back on your own machine

Unzip into `results/`, then re-derive every table and figure without touching
a GPU or the API:

```bat
python -m src.run --audio_dir audio --out results ^
  --transcripts results/transcripts_local.jsonl --skip_transcribe
```

Record in `results/README.md` which backend and which date produced the
numbers, so the run is reproducible.